<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Реализация класса Invoice</h2>

----

### Вариант задания №10


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Invoice в C#, который будет представлять информацию о фактурах за поставленные товары или оказанные услуги. На основе этого класса разработать 2-3 производных класса, демонстрирующих принципы наследования и полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и методы, а также переопределены некоторые методы базового класса для
демонстрации полиморфизма.
Требования к базовому классу Invoice:

• Атрибуты: Номер фактуры (InvoiceNumber), Дата выдачи (IssueDate), Общая
сумма (TotalAmount).

• Методы:

o CalculateTotal(): метод для расчета общей суммы по фактуре.

o AddLine(LineItem lineItem): метод для добавления позиции в фактуру.

o RemoveLine(LineItem lineItem): метод для удаления позиции из фактуры.

Требования к производным классам:

1. ТоварнаяФактура (GoodsInvoice): Должна содержать дополнительные атрибуты, такие как Дата поставки (SupplyDate). Метод AddLine() должен быть переопределен для добавления информации о дате поставки товара при добавлении позиции.

2. УслуговаяФактура (ServiceInvoice): Должна содержать дополнительные атрибуты, такие как Дата оказания услуги (ServiceDate). Метод RemoveLine() должен быть переопределен для добавления информации о причине аннулирования услуги при удалении позиции.

3. КомбинированнаяФактура (CombinedInvoice) (если требуется третий класс): Должна содержать дополнительные атрибуты, такие как Наличие возврата (ReturnAllowed). Метод CalculateTotal() должен быть переопределен для учета возможного возврата товара или услуги при расчете общей суммы.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
using System;
using System.Collections.Generic;
using System.Globalization;

// --- ИСПОЛНЯЕМАЯ ЧАСТЬ ---

Console.WriteLine("=== Создание объектов ===");

// Создаем обычную товарную фактуру
var goodsInv = new GoodsInvoice(
    "INV-001",
    DateTime.Today,
    "Client-A",
    DateTime.Today.AddDays(5)
);

// Создаем фактуру услуги
var serviceInv = new ServiceInvoice(
    "SRV-001",
    DateTime.Today,
    "Client-B",
    DateTime.Today.AddDays(1)
);

// Создаем фактуру возврата
var returnInv = new ReturnInvoice(
    "RET-001",
    DateTime.Today,
    "Client-C",
    true // ReturnAllowed = true
);
returnInv.ReturnComment = "Не подошел размер";

// ==================================================================
// ПОДПИСКА НА СОБЫТИЯ (ИСПРАВЛЕНИЕ: подписываемся на экземпляры)
// ==================================================================

// Подписываемся на события каждого экземпляра
goodsInv.TotalAmountChanged += HandleTotalChangeLog;
goodsInv.TotalAmountChanged += (source, msg) => Console.WriteLine($"[{DateTime.Now:HH:mm:ss}] Лямбда Goods: {msg}");

serviceInv.TotalAmountChanged += HandleTotalChangeLog;
serviceInv.TotalAmountChanged += (source, msg) => Console.WriteLine($"[{DateTime.Now:HH:mm:ss}] Лямбда Service: {msg}");

returnInv.TotalAmountChanged += HandleTotalChangeLog;
returnInv.TotalAmountChanged += (source, msg) => Console.WriteLine($"[{DateTime.Now:HH:mm:ss}] Лямбда Return: {msg}");

// Добавление позиций (вызовет пересчет и события)
Console.WriteLine("\n--- Добавление позиций ---");
goodsInv.AddLine(new LinItem { ItemName = "Laptop XPS", Price = 1000, Quantity = 1 });
goodsInv.AddLine(new LinItem { ItemName = "Mouse", Price = 50, Quantity = 2 });

serviceInv.AddLine(new LinItem { ItemName = "IT Consulting", Price = 200, Quantity = 1 });

returnInv.AddLine(new LinItem { ItemName = "Shoes Nike", Price = 150, Quantity = 1 });

Console.WriteLine("\n=== Отображение информации (Полиморфизм) ===");
goodsInv.DisplayDetails();
serviceInv.DisplayDetails();
returnInv.DisplayDetails();

Console.WriteLine("\n=== Изменение данных (Триггеры событий) ===");

// Удаляем позицию из услугной фактуры
// Благодаря переопределению Equals в LinItem, удаление сработает корректно
LinItem consultingItem = new LinItem { ItemName = "IT Consulting", Price = 200, Quantity = 1 };
Console.WriteLine("\nПопытка удалить позицию из ServiceInvoice:");
serviceInv.RemoveLine(consultingItem);

// Явный пересчет вызовет событие
Console.WriteLine("\nЯвный пересчет сумм:");
goodsInv.CalculateTotal();
returnInv.CalculateTotal();

Console.WriteLine("\n=== Финальное состояние ===");
goodsInv.DisplayDetails();
serviceInv.DisplayDetails();
returnInv.DisplayDetails();

// Статический метод-обработчик события
static void HandleTotalChangeLog(Invoice source, string message)
{
    Console.WriteLine($">>> СИСТЕМНЫЙ ЛОГ: [ID={source.InvoiceNumber}] {message}");
}

// --- КЛАССЫ ---

// Вспомогательный класс позиции
public class LinItem
{
    public string ItemName { get; set; }
    public double Price { get; set; }
    public int Quantity { get; set; }

    public override string ToString()
    {
        return $"{ItemName}: {Quantity} x {Price:C} = {(Price * Quantity):C}";
    }

    // ИСПРАВЛЕНИЕ: Переопределяем Equals и GetHashCode для корректной работы Contains/Remove
    public override bool Equals(object obj)
    {
        return obj is LinItem other &&
               ItemName == other.ItemName &&
               Price == other.Price &&
               Quantity == other.Quantity;
    }

    public override int GetHashCode()
    {
        return HashCode.Combine(ItemName, Price, Quantity);
    }
}

// Делегат для события
public delegate void TotalAmountChangedDelegate(Invoice source, string message);

// Базовый класс Invoice
public class Invoice
{
    public string InvoiceNumber { get; set; }
    public DateTime IssueDate { get; set; }
    public double TotalAmount { get; set; }
    public string CustomerId { get; set; }

    // ИСПРАВЛЕНИЕ CS0122: Изменено private на protected для доступа в наследниках
    protected List<LinItem> _lineItems;

    // ИСПРАВЛЕНИЕ CS0117: Событие переименовано в TotalAmountChanged
    public event TotalAmountChangedDelegate TotalAmountChanged;

    public Invoice(string number, DateTime date, string customerId)
    {
        InvoiceNumber = number;
        IssueDate = date;
        CustomerId = customerId;
        _lineItems = new List<LinItem>();
        TotalAmount = 0;
    }

    public virtual void CalculateTotal()
    {
        TotalAmount = 0;
        foreach (var item in _lineItems)
        {
            TotalAmount += item.Price * item.Quantity;
        }
        InvokeTotalAmountChanged($"Сумма пересчитана: {TotalAmount:C}");
    }

    public virtual void AddLine(LinItem linItem)
    {
        _lineItems.Add(linItem);
        CalculateTotal();
    }

    public virtual void RemoveLine(LinItem linItem)
    {
        if (_lineItems.Contains(linItem))
        {
            _lineItems.Remove(linItem);
            CalculateTotal();
        }
        else
        {
            Console.WriteLine($"[Invoice] Позиция '{linItem.ItemName}' не найдена для удаления.");
        }
    }

    public virtual void DisplayDetails()
    {
        Console.WriteLine($"\n--- Фактура #{InvoiceNumber} ---");
        Console.WriteLine($"Дата: {IssueDate:dd.MM.yyyy} | Клиент: {CustomerId}");
        Console.WriteLine("Позиции:");
        foreach (var item in _lineItems)
        {
            Console.WriteLine($"  {item}");
        }
        Console.WriteLine($"ИТОГО: {TotalAmount:C}");
    }

    protected virtual void InvokeTotalAmountChanged(string message)
    {
        TotalAmountChanged?.Invoke(this, message);
    }
}

// Товарная фактура
public class GoodsInvoice : Invoice
{
    public DateTime SupplyDate { get; set; }

    public GoodsInvoice(string number, DateTime issueDate, string customerId, DateTime supplyDate)
        : base(number, issueDate, customerId)
    {
        SupplyDate = supplyDate;
    }

    public override void AddLine(LinItem linItem)
    {
        Console.WriteLine("[GoodsInvoice] Добавление товара...");
        base.AddLine(linItem);
    }

    public override void DisplayDetails()
    {
        Console.WriteLine("\n--- ТОВАРНАЯ ФАКТУРА ---");
        base.DisplayDetails();
        Console.WriteLine($"Дата поставки: {SupplyDate:dd.MM.yyyy}");
    }
}

// Услуговая фактура
public class ServiceInvoice : Invoice
{
    public DateTime ServiceDate { get; set; }
    private string LastRemovalReason { get; set; } = string.Empty;

    public ServiceInvoice(string number, DateTime issueDate, string customerId, DateTime serviceDate)
        : base(number, issueDate, customerId)
    {
        ServiceDate = serviceDate;
    }

    public override void RemoveLine(LinItem linItem)
    {
        string reason = "Отмена услуги без указания причины";
        if (linItem.ItemName == "Консультация" || linItem.ItemName == "IT Consulting")
            reason = "Услуга отменена по запросу клиента";

        LastRemovalReason = reason;
        Console.WriteLine($"[ServiceInvoice] Удаление позиции '{linItem.ItemName}' с причиной: {reason}");
        base.RemoveLine(linItem);
    }

    public override void DisplayDetails()
    {
        Console.WriteLine("\n--- УСЛУГОВАЯ ФАКТУРА ---");
        base.DisplayDetails();
        Console.WriteLine($"Дата услуги: {ServiceDate:dd.MM.yyyy}");
    }
}

// Фактура возврата
public class ReturnInvoice : Invoice
{
    public bool ReturnAllowed { get; set; }
    public string ReturnComment { get; set; }

    public ReturnInvoice(string number, DateTime issueDate, string customerId, bool allowed)
        : base(number, issueDate, customerId)
    {
        ReturnAllowed = allowed;
    }

    public override void CalculateTotal()
    {
        double sum = 0;
        // ИСПРАВЛЕНИЕ: _lineItems теперь доступен благодаря protected
        foreach (var item in _lineItems)
        {
            if (ReturnAllowed)
                sum -= (item.Price * item.Quantity);
            else
                sum += (item.Price * item.Quantity);
        }
        TotalAmount = sum;

        if (ReturnAllowed)
            InvokeTotalAmountChanged($"Сформирован возврат на сумму {Math.Abs(TotalAmount):C}");
        else
            InvokeTotalAmountChanged($"Пересчет завершен: {TotalAmount:C}");
    }

    public override void DisplayDetails()
    {
        Console.WriteLine("\n--- ФАКТУРА ВОЗВРАТА ---");
        base.DisplayDetails();
        Console.WriteLine($"Возврат разрешен: {ReturnAllowed}");
        if (!string.IsNullOrEmpty(ReturnComment))
            Console.WriteLine($"Комментарий: {ReturnComment}");
    }
}

=== Создание объектов ===

--- Добавление позиций ---
[GoodsInvoice] Добавление товара...
>>> СИСТЕМНЫЙ ЛОГ: [ID=INV-001] Сумма пересчитана: 1 000,00 ¤
[08:28:45] Лямбда Goods: Сумма пересчитана: 1 000,00 ¤
[GoodsInvoice] Добавление товара...
>>> СИСТЕМНЫЙ ЛОГ: [ID=INV-001] Сумма пересчитана: 1 100,00 ¤
[08:28:45] Лямбда Goods: Сумма пересчитана: 1 100,00 ¤
>>> СИСТЕМНЫЙ ЛОГ: [ID=SRV-001] Сумма пересчитана: 200,00 ¤
[08:28:45] Лямбда Service: Сумма пересчитана: 200,00 ¤
>>> СИСТЕМНЫЙ ЛОГ: [ID=RET-001] Сформирован возврат на сумму 150,00 ¤
[08:28:45] Лямбда Return: Сформирован возврат на сумму 150,00 ¤

=== Отображение информации (Полиморфизм) ===

--- ТОВАРНАЯ ФАКТУРА ---

--- Фактура #INV-001 ---
Дата: 07.04.2026 | Клиент: Client-A
Позиции:
  Laptop XPS: 1 x 1 000,00 ¤ = 1 000,00 ¤
  Mouse: 2 x 50,00 ¤ = 100,00 ¤
ИТОГО: 1 100,00 ¤
Дата поставки: 12.04.2026

--- УСЛУГОВАЯ ФАКТУРА ---

--- Фактура #SRV-001 ---
Дата: 07.04.2026 | Клиент: Client-B
Позиции:
  IT Consulting: 1 x 200,00 ¤ = 